In [1]:
import os

In [2]:
%pwd

'c:\\Users\\2021ICTS28\\Desktop\\wine_quality_end_to_end_project\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'c:\\Users\\2021ICTS28\\Desktop\\wine_quality_end_to_end_project'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    alpha: float
    l1_ration: float
    target_col: str

In [6]:
from src.wineProject.utils.common import read_yaml, create_directories
from src.wineProject.constants import *

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path = CONFIG_FILE_PATH,
        params_file_path = PARAMS_FILE_PATH,
        schema_file_path = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)
        
        create_directories([self.config.artifacts_root])
        
    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN
        
        model_trainer_config = ModelTrainerConfig(
            root_dir = config.root_dir,
            train_data_path = config.train_data_path,
            test_data_path = config.test_data_path,
            model_name = config.model_name,
            alpha = params.alpha,
            l1_ration = params.l1_ratio,
            target_col = schema.name
        )
        
        return model_trainer_config
        

In [8]:
import pandas as pd
import os
from src.wineProject import logger
from sklearn.linear_model import ElasticNet
import joblib

In [9]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config
        
    def train_model(self):
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)
        logger.info(f"Train data and test data are loaded successfully.")
        
        train_x = train_data.drop(self.config.target_col, axis=1)
        train_y = train_data[self.config.target_col]
        
        test_x = test_data.drop(self.config.target_col, axis=1)
        test_y = test_data[self.config.target_col]
        logger.info(f"Splitting of data into dependent and independent features is done successfully.")
        
        model = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ration, random_state=42)
        model.fit(train_x, train_y)
        logger.info(f"Model training is done successfully.")
        
        model_dir = os.path.join(self.config.root_dir, self.config.model_name)
        create_directories([self.config.root_dir])
        joblib.dump(model, model_dir)
        logger.info(f"Model is saved at {model_dir} successfully.")
        

In [10]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train_model()
except Exception as e:  
    logger.exception(e)

[2026-03-21 08:25:57,132: INFO: common]: yaml file: config\config.yaml loaded successfully
[2026-03-21 08:25:57,132: INFO: common]: yaml file: params.yaml loaded successfully
[2026-03-21 08:25:57,132: INFO: common]: yaml file: schema.yaml loaded successfully
[2026-03-21 08:25:57,132: INFO: common]: created directory at: artifacts
[2026-03-21 08:25:57,145: INFO: 1202825799]: Train data and test data are loaded successfully.
[2026-03-21 08:25:57,149: INFO: 1202825799]: Splitting of data into dependent and independent features is done successfully.
[2026-03-21 08:25:57,151: INFO: 1202825799]: Model training is done successfully.
[2026-03-21 08:25:57,152: INFO: common]: created directory at: artifacts/model_trainer
[2026-03-21 08:25:57,154: INFO: 1202825799]: Model is saved at artifacts/model_trainer\model.joblib successfully.
